Imports


In [ ]:
import numpy as np
import string

Tensor Class and Methods

In [ ]:
class Tensor:
    def __init__(self, data: np.ndarray | list, calculateGrad: bool = False):
        self.data = data if isinstance(data, np.ndarray) else np.array(data, dtype=np.double)

        self.shape = np.shape(self.data)
        self.ndim = np.ndim(self.data)
        self.grad = np.zeros(self.shape)
        self.children: list[Tensor] = []
        self.calculateGrad = calculateGrad
        self._backward = lambda: None

    def __repr__(self):
        return f"Tensor(data=\n{self.data})"

    def backward(self):
        topo = []
        visited = set()
        def build_topo(t: Tensor):
            if t not in visited:
                visited.add(t)
                for child in t.children:
                    build_topo(child)
                topo.append(t)

        build_topo(self)
        for t in reversed(topo):
            t._backward()

    # --- Rechenoperationen ---
    # Komponentenweise Addition
    def __add__(self, other):
        out = Tensor(self.data + other.data)
        out.children += [self, other]
        out.calculateGrad = self.calculateGrad | other.calculateGrad
        def _backward():
            self.grad += out.grad if self.calculateGrad else 0
            other.grad += out.grad if other.calculateGrad else 0

        out._backward = _backward
        return out

    # Matrix-Matrix Multiplikation (Max. 2-Achsig)
    def __matmul__(self, other):
        left  = "ij" if self.ndim  == 2 else "j"
        right = "jk" if other.ndim == 2 else "j"
        out   = ("i" if self.ndim  == 2 else "") + ("k" if other.ndim == 2 else "")
        schema = f"{left},{right}->{out}"

        out = contraction([self, other], schema) 
        out.children += [self, other]
        out.calculateGrad = self.calculateGrad | other.calculateGrad
        def _backward():
            self.grad += _partial_gradient([self.data, other.data], schema, 0, out.grad) if self.calculateGrad else 0
            other.grad += _partial_gradient([self.data, other.data], schema, 1, out.grad) if other.calculateGrad else 0

        out._backward = _backward
        return out
    
    # Komponentenweise Multiplikation
    def __mul__(self, other):
        out = Tensor(self.data * other.data)
        out.children += [self, other]
        out.calculateGrad = self.calculateGrad | other.calculateGrad
        def _backward():
            self.grad += other.data * out.grad if self.calculateGrad else 0
            other.grad += self.data * out.grad if other.calculateGrad else 0

        out._backward = _backward
        return out


Tensor Operations & Automatic Jacobian/ Gradient

In [ ]:
def insert(tensor: Tensor, insertion: tuple[int, ...]):
    return Tensor(np.expand_dims(tensor.data, insertion))

def stretch(tensor: Tensor, insertion: tuple[int, ...], length: tuple[int, ...]) -> Tensor:
    array = np.expand_dims(tensor.data, insertion)

    shape = list(tensor.shape)
    for i, val in zip(insertion, length): 
        shape.insert(i, val)
    out = Tensor(np.broadcast_to(array, shape))
    return out

def contraction(tensors: list[Tensor], schema: str):
    return Tensor(np.einsum(schema, *[t.data for t in tensors]))

def partial_jacobian(tensors: list[Tensor], schema: str, targetPos: int):
    # Determine unused letters in schema
    lhs, rhs = schema.replace(" ", "").split("->")
    inds = lhs.split(",")
    l_used, r_used = set(lhs.replace(",", "")), set(rhs)
    unused = list(c for c in string.ascii_letters if c not in l_used | r_used)
    unused.sort()
    
    # Remove target tensor from indices 
    _tensors = list(tensors)
    target = _tensors.pop(targetPos)
    targetInd = inds.pop(targetPos)

    # Construct in- and output indices
    outInd = rhs
    trailing_dims = {}
    for pos, (i, p) in enumerate(zip(targetInd, unused)):
        # Construct output index
        outInd += p
        trailing_dims[p] = target.shape[pos]

        # Modify input indices
        # Case: Index* in f-Block
        if i in rhs:
            axis_length = target.shape[pos]
            _tensors.append(Tensor(np.identity(axis_length)))
            inds.append(i + p)
        
        # Case: Index* not in f-Block
        else:
            inds = [ind.replace(i, p) for ind in inds]

    # Determine indices and their positions not occuring in input
    inds_set = set("".join(inds))
    missing = set(outInd) - inds_set

    insertions = tuple(pos for pos, i in enumerate(outInd) if i in missing)
    occuring_out = "".join([i for i in outInd if i in inds_set])

    # Calculate contraction with occuring indices
    if inds: # to prevent empty einsum
        occuring_result = np.einsum(f"{",".join(inds)} -> {occuring_out}", *[t.data for t in _tensors])
    else:
        occuring_result = np.array(1.0)

    # Broadcast missing indices to right length
    final_shape = (trailing_dims[p] if p in trailing_dims else np.shape(occuring_result)[pos] for pos, p in enumerate(outInd))

    inserted_result = np.expand_dims(occuring_result, insertions)
    final_result = np.broadcast_to(inserted_result, final_shape)

    return Tensor(final_result)

def partial_gradient(tensors: list[Tensor], schema: str, targetPos: int, outGrad: Tensor):
    lhs, rhs = schema.replace(" ", "").split("->")
    inds = lhs.split(",")

    # Remove target tensor from indices 
    _tensors = list(tensors)
    target = _tensors.pop(targetPos)
    targetInd = inds.pop(targetPos)

    # Construct in- and output indices
    outInd = targetInd
    inds.append(rhs)
    _tensors.append(outGrad)
    
    # Determine indices and their positions not occuring in input
    inds_set = set("".join(inds))
    missing = set(outInd) - inds_set

    insertions = tuple(pos for pos, i in enumerate(outInd) if i in missing)
    occuring_out = "".join([i for i in outInd if i in inds_set])

    # Calculate contraction with occuring indices
    if inds: # to prevent empty einsum
        occuring_result = np.einsum(f"{",".join(inds)} -> {occuring_out}", *[t.data for t in _tensors])
    else:
        occuring_result = np.array(1.0)

    # Broadcast missing indices to right length
    final_shape = target.shape

    inserted_result = np.expand_dims(occuring_result, insertions)
    final_result = np.broadcast_to(inserted_result, final_shape)

    return Tensor(final_result)

def _partial_gradient(tensors: list[np.ndarray], schema: str, targetPos: int, outGrad: np.ndarray):
    lhs, rhs = schema.replace(" ", "").split("->")
    inds = lhs.split(",")

    # Remove target tensor from indices 
    _tensors = list(tensors)
    target = _tensors.pop(targetPos)
    targetInd = inds.pop(targetPos)

    # Construct in- and output indices
    outInd = targetInd
    inds.append(rhs)
    _tensors.append(outGrad)
    
    # Determine indices and their positions not occuring in input
    inds_set = set("".join(inds))
    missing = set(outInd) - inds_set

    insertions = tuple(pos for pos, i in enumerate(outInd) if i in missing)
    occuring_out = "".join([i for i in outInd if i in inds_set])

    # Calculate contraction with occuring indices
    if inds: # to prevent empty einsum
        occuring_result = np.einsum(f"{",".join(inds)} -> {occuring_out}", *[t for t in _tensors])
    else:
        occuring_result = np.array(1.0)

    # Broadcast missing indices to right length
    final_shape = np.shape(target)

    inserted_result = np.expand_dims(occuring_result, insertions)
    final_result = np.broadcast_to(inserted_result, final_shape)

    return final_result

Testing

In [ ]:
rng = np.random.default_rng(71116)

A = Tensor(np.arange(9.0).reshape([3,3]), True)
B = Tensor(rng.random([3,3], dtype=np.float32))
x = Tensor([1,10,100], True)
y = Tensor([1,2,3]) 

# arguments = [A,B]
# schema = "ij,jk->ik"
# outGradient = partial_jacobian([contraction(arguments, schema)], "ij->", 0)

# print("Map: ", contraction(arguments, schema))

# print("\nPartial jacobian i.r.t. argument 0:", partial_jacobian(arguments, schema, 0))
# print("\nPartial jacobian i.r.t. argument 1:", partial_jacobian(arguments, schema, 1))

# print("\nPartial gradient i.r.t. argument 0:", partial_gradient(arguments, schema, 0, outGradient))
# print("\nPartial gradient i.r.t. argument 1:", partial_gradient(arguments, schema, 1, outGradient))

C = A @ x 
# z = C @ C
Z = (A @ x) @ (A @ x)
Z.grad = np.array(1)

Z.backward()

# print(z)
print(A.grad)